# English → Chinese Translation
### Using `Qwen/Qwen2.5-7B-Instruct` on a free Colab T4 GPU

**Before running:**
1. Go to **Runtime → Change runtime type → T4 GPU → Save**
2. Run all cells top to bottom
3. Upload your `chinese_sub_corpus.json` when prompted in Cell 3
4. Download the translated file from Cell 6

## 1 — Check GPU

In [ ]:
import torch

if torch.cuda.is_available():
    print(f"GPU : {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected. Go to Runtime > Change runtime type > T4 GPU.")

## 2 — Install dependencies

In [ ]:
!pip install -q transformers accelerate tqdm

## 3 — Upload your JSON file

In [ ]:
from google.colab import files

print("Upload your chinese_sub_corpus.json file:")
uploaded = files.upload()   # file picker will appear

INPUT_FILENAME = list(uploaded.keys())[0]
print(f"\nUploaded: {INPUT_FILENAME}")

## 4 — Load Qwen model

> **Note:** Qwen2.5-7B is ~14 GB in fp16. The T4 has 16 GB VRAM so it fits,
> but leave `BATCH_SIZE` at 4–8 to avoid running out of memory.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

print(f"Loading {MODEL_NAME} onto {DEVICE} ...")
print("(This will download ~14 GB the first time — expect 5–10 minutes)\n")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model     = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,   # fp16 to fit in T4 VRAM
    device_map="auto",           # automatically places layers on GPU
    trust_remote_code=True,
)
model.eval()
print("\nModel ready.")

## 5 — Translate

In [ ]:
import json, os
from tqdm.notebook import tqdm

# ── Config ────────────────────────────────────────────────────────────────────
BATCH_SIZE       = 4    # keep low for T4 (16 GB VRAM); raise to 8 if no OOM
MAX_NEW_TOKENS   = 256  # max tokens in the Chinese output
CHECKPOINT_FILE  = "translation_checkpoint.json"
CHECKPOINT_EVERY = 10   # save every N batches (crash recovery)

# ── Load data ─────────────────────────────────────────────────────────────────
with open(INPUT_FILENAME, "r", encoding="utf-8") as f:
    raw = f.read().strip()

try:
    data = json.loads(raw)
    if not isinstance(data, list):
        data = [data]
except json.JSONDecodeError:
    data = [json.loads(line) for line in raw.splitlines() if line.strip()]

indices = [i for i, item in enumerate(data) if item.get("text")]
texts   = [data[i]["text"] for i in indices]
print(f"Records to translate: {len(texts)}")

# ── Resume from checkpoint if available ───────────────────────────────────────
results     = []
start_index = 0

if os.path.exists(CHECKPOINT_FILE):
    with open(CHECKPOINT_FILE, "r") as f:
        results = json.load(f)
    start_index = len(results)
    print(f"Resuming from checkpoint: {start_index}/{len(texts)} already done.")

remaining = texts[start_index:]

# ── Translate ─────────────────────────────────────────────────────────────────
with torch.inference_mode():
    for batch_num, i in enumerate(tqdm(range(0, len(remaining), BATCH_SIZE), desc="Translating")):
        batch = remaining[i : i + BATCH_SIZE]

        # Wrap each text in a translation prompt
        prompts = [
            f"Translate the following English text to Chinese. Output only the translation, nothing else.\n\n{t}"
            for t in batch
        ]

        tokens = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512,
        ).to(DEVICE)

        out = model.generate(
            **tokens,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,                       # greedy — deterministic & faster
            pad_token_id=tokenizer.eos_token_id,
        )

        # Causal LMs echo the prompt — slice it off to keep only the translation
        input_len = tokens["input_ids"].shape[1]
        decoded   = tokenizer.batch_decode(out[:, input_len:], skip_special_tokens=True)
        results.extend(decoded)

        # Periodic checkpoint save
        if (batch_num + 1) % CHECKPOINT_EVERY == 0:
            with open(CHECKPOINT_FILE, "w") as f:
                json.dump(results, f, ensure_ascii=False)

print(f"\nTranslation complete: {len(results)} records.")

## 6 — Save & download

In [ ]:
OUTPUT_FILENAME = "chinese_sub_corpus_translated.json"

# Write translations back into the original records
for idx, translation in zip(indices, results):
    data[idx]["text_zh"] = translation   # original "text" field is preserved

with open(OUTPUT_FILENAME, "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

# Clean up checkpoint file
if os.path.exists(CHECKPOINT_FILE):
    os.remove(CHECKPOINT_FILE)

# Download to your local machine
files.download(OUTPUT_FILENAME)
print(f"Downloaded: {OUTPUT_FILENAME}")